# 08 — Analytics: 10 Queries de Portfolio

Demonstra o poder do modelo dimensional com 10 queries analíticas usando **DuckDB SQL**.

**Técnicas DuckDB exclusivas demonstradas:**
- `QUALIFY` — filtrar window functions sem subquery externa
- `PIVOT` — tabela dinâmica nativa
- Window functions: `RANK`, `LAG`, `LEAD`, `NTILE`, `SUM OVER`
- `LIST_AGG` / `STRING_AGG` com ORDER BY inline

| # | Query | Técnica DuckDB |
|---|-------|----------------|
| 1 | Receita bruta vs líquida por categoria | GROUP BY + métricas |
| 2 | Top 10 clientes por NetRevenue | RANK + QUALIFY |
| 3 | Performance de entrega por Shipper | Accumulating Snapshot |
| 4 | Análise SCD2: receita pelo país histórico | SCD2 awareness |
| 5 | Histórico de preço por produto | Versões SCD2 |
| 6 | Hierarquia de funcionários: receita por gestor | Flattened hierarchy |
| 7 | Sazonalidade: NetRevenue por mês com LAG | LAG para variação MoM |
| 8 | Produtos com reposição necessária | Último snapshot |
| 9 | Desconto médio por categoria | NTILE para segmentação |
| 10 | Pedidos por país com lead time médio | Accumulating Snapshot |

In [1]:
import sys, os
sys.path.insert(0, os.getcwd())
from utils import get_conn, DB_PATH, DATA_DIR

conn = get_conn()
print(f"Conectado: {DB_PATH}")

Conectado: /workspace/pf_northwind/duckdb/northwind_dw.duckdb


In [2]:
# ============================================================
# Query 1: Receita bruta vs líquida por categoria de produto
# ============================================================
print("Query 1 — Receita por Categoria")
print(conn.execute("""
    SELECT
        p.CategoryName,
        COUNT(DISTINCT f.OrderID)                           AS Pedidos,
        ROUND(SUM(f.GrossRevenue), 2)                       AS GrossRevenue,
        ROUND(SUM(f.NetRevenue), 2)                         AS NetRevenue,
        ROUND(SUM(f.GrossRevenue - f.NetRevenue), 2)        AS DescontoTotal,
        ROUND(AVG(f.Discount) * 100, 2)                     AS DescontoMedioPct
    FROM gold.FactSales f
    JOIN gold.DimProduct p ON f.ProductSK = p.ProductSK
    GROUP BY p.CategoryName
    ORDER BY NetRevenue DESC
""").fetchdf().to_string(index=False))

Query 1 — Receita por Categoria
  CategoryName  Pedidos  GrossRevenue  NetRevenue  DescontoTotal  DescontoMedioPct
     Beverages      354     286526.95   267868.18       18658.77              6.19
Dairy Products      303     251330.50   234507.28       16823.22              5.34
   Confections      295     177099.10   167357.22        9741.88              5.69
  Meat/Poultry      161     178188.80   163022.36       15166.44              6.45
       Seafood      291     141623.09   131261.74       10361.35              6.02
    Condiments      193     113694.75   106047.08        7647.67              5.26
       Produce      129     105268.60    99984.58        5284.02              4.54
Grains/Cereals      182     100726.80    95744.59        4982.21              4.53


In [3]:
# ============================================================
# Query 2: Top 10 clientes por NetRevenue
# Técnica DuckDB: QUALIFY RANK() — sem subquery externa
# ============================================================
print("Query 2 — Top 10 Clientes (QUALIFY RANK)")
print(conn.execute("""
    SELECT
        c.CompanyName,
        c.Country,
        COUNT(DISTINCT f.OrderID)           AS Pedidos,
        ROUND(SUM(f.NetRevenue), 2)         AS NetRevenue,
        ROUND(AVG(f.NetRevenue), 2)         AS TicketMedio,
        RANK() OVER (ORDER BY SUM(f.NetRevenue) DESC) AS Rank
    FROM gold.FactSales f
    JOIN gold.DimCustomer c ON f.CustomerSK = c.CustomerSK
    GROUP BY c.CompanyName, c.Country
    QUALIFY RANK() OVER (ORDER BY SUM(f.NetRevenue) DESC) <= 10
    ORDER BY Rank
""").fetchdf().to_string(index=False))

Query 2 — Top 10 Clientes (QUALIFY RANK)
                 CompanyName Country  Pedidos  NetRevenue  TicketMedio  Rank
                  QUICK-Stop Germany       28   110277.30      1282.29     1
                Ernst Handel Austria       30   104874.98      1028.19     2
          Save-a-lot Markets     USA       31   104361.95       899.67     3
  Rattlesnake Canyon Grocery     USA       18    51097.80       719.69     4
Hungry Owl All-Night Grocers Ireland       19    49979.90       908.73     5
               Hanari Carnes  Brazil       14    32841.37      1026.29     6
             Königlich Essen Germany       14    30908.38       792.52     7
              Folk och fä HB  Sweden       19    29567.56       657.06     8
              Mère Paillarde  Canada       13    28872.19       902.26     9
        White Clover Markets     USA       14    27363.60       684.09    10


In [4]:
# ============================================================
# Query 3: Performance de entrega por Shipper (Accumulating Snapshot)
# ============================================================
print("Query 3 — Performance de Entrega por Shipper")
print(conn.execute("""
    SELECT
        s.CompanyName                                                       AS Shipper,
        COUNT(*)                                                            AS TotalEntregues,
        SUM(CASE WHEN f.IsLate = FALSE THEN 1 ELSE 0 END)                  AS OnTime,
        SUM(CASE WHEN f.IsLate = TRUE  THEN 1 ELSE 0 END)                  AS Atrasados,
        ROUND(100.0 * SUM(CASE WHEN f.IsLate = FALSE THEN 1 ELSE 0 END)
              / COUNT(*), 1)                                               AS OnTimePct,
        ROUND(AVG(f.DaysToShip::DOUBLE), 1)                               AS MediaDiasEnvio
    FROM gold.FactOrderFulfillment f
    JOIN gold.DimShipper s ON f.ShipperSK = s.ShipperSK
    WHERE f.ShippedDateKey IS NOT NULL
    GROUP BY s.CompanyName
    ORDER BY OnTimePct DESC
""").fetchdf().to_string(index=False))

Query 3 — Performance de Entrega por Shipper
         Shipper  TotalEntregues  OnTime  Atrasados  OnTimePct  MediaDiasEnvio
Federal Shipping             250     240         10       96.0            48.2
  Speedy Express             245     233         12       95.1             8.6
  United Package             315     299         16       94.9             9.2


In [5]:
# ============================================================
# Query 4: Análise SCD2 — receita pelo país do cliente NO MOMENTO DA VENDA
# Demonstra o valor do SCD2: perfil histórico, não o atual
# ============================================================
print("Query 4 — Receita por País Histórico do Cliente (SCD2 awareness)")
print(conn.execute("""
    SELECT
        c.Country                               AS PaisNaMomentoVenda,
        d.Year                                  AS Ano,
        COUNT(DISTINCT f.OrderID)               AS Pedidos,
        ROUND(SUM(f.NetRevenue), 2)             AS NetRevenue,
        RANK() OVER (
            PARTITION BY d.Year
            ORDER BY SUM(f.NetRevenue) DESC
        )                                       AS Rank
    FROM gold.FactSales f
    JOIN gold.DimCustomer c ON f.CustomerSK = c.CustomerSK
    JOIN gold.DimDate d     ON f.OrderDateKey = d.DateKey
    GROUP BY c.Country, d.Year
    QUALIFY RANK() OVER (PARTITION BY d.Year ORDER BY SUM(f.NetRevenue) DESC) <= 5
    ORDER BY d.Year, Rank
""").fetchdf().to_string(index=False))

Query 4 — Receita por País Histórico do Cliente (SCD2 awareness)
PaisNaMomentoVenda  Ano  Pedidos  NetRevenue  Rank
               USA 1996       23    38105.67     1
           Germany 1996       24    35407.14     2
           Austria 1996        8    25601.34     3
            Brazil 1996       13    20148.82     4
            France 1996       15    17372.76     5
           Germany 1997       64   117320.16     1
               USA 1997       60   114845.26     2
           Austria 1997       21    57401.84     3
            France 1997       39    45263.38     4
            Brazil 1997       42    41941.19     5
               USA 1998       39    92633.67     1
           Germany 1998       34    77557.32     2
           Austria 1998       11    45000.65     3
            Brazil 1998       28    44835.77     4
                UK 1998       16    22623.53     5


In [6]:
# ============================================================
# Query 5: Histórico de preço por produto — versões SCD2
# ============================================================
print("Query 5 — Histórico de Preço por Produto (versões SCD2)")
print(conn.execute("""
    SELECT
        p.ProductName,
        p.CategoryName,
        p.UnitPrice       AS PrecoNaVersao,
        p.ValidFrom,
        p.ValidTo,
        COUNT(f.SalesSK)                            AS Vendas,
        ROUND(COALESCE(SUM(f.NetRevenue), 0), 2)    AS ReceitaNaVersao
    FROM gold.DimProduct p
    LEFT JOIN gold.FactSales f ON f.ProductSK = p.ProductSK
    GROUP BY p.ProductName, p.CategoryName, p.UnitPrice, p.ValidFrom, p.ValidTo
    ORDER BY p.ProductName, p.ValidFrom
    LIMIT 20
""").fetchdf().to_string(index=False))

Query 5 — Histórico de Preço por Produto (versões SCD2)
                 ProductName   CategoryName  PrecoNaVersao  ValidFrom    ValidTo  Vendas  ReceitaNaVersao
                Alice Mutton   Meat/Poultry          39.00 1900-01-01 9999-12-31      37         32698.38
               Aniseed Syrup     Condiments          10.00 1900-01-01 9999-12-31      12          3044.00
            Boston Crab Meat        Seafood          18.40 1900-01-01 9999-12-31      41         17910.63
           Camembert Pierrot Dairy Products          34.00 1900-01-01 9999-12-31      51         46825.48
            Carnarvon Tigers        Seafood          62.50 1900-01-01 9999-12-31      27         29171.87
                        Chai      Beverages          18.00 1900-01-01 9999-12-31      38         12788.10
                       Chang      Beverages          19.00 1900-01-01 9999-12-31      44         16355.96
            Chartreuse verte      Beverages          18.00 1900-01-01 9999-12-31      30        

In [7]:
# ============================================================
# Query 6: Hierarquia de funcionários — receita por gestor
# Demonstra a hierarquia achatada do DimEmployee
# ============================================================
print("Query 6 — Hierarquia de Funcionários: Receita por Gestor")
print(conn.execute("""
    SELECT
        COALESCE(e.ManagerName, '(Presidente)')  AS Gestor,
        e.FullName                               AS Funcionario,
        e.Title,
        COUNT(f.SalesSK)                         AS Transacoes,
        ROUND(SUM(f.NetRevenue), 2)              AS NetRevenue,
        ROUND(AVG(f.NetRevenue), 2)              AS TicketMedio
    FROM gold.FactSales f
    JOIN gold.DimEmployee e ON f.EmployeeSK = e.EmployeeSK
    GROUP BY e.ManagerName, e.FullName, e.Title
    ORDER BY Gestor, NetRevenue DESC
""").fetchdf().to_string(index=False))

Query 6 — Hierarquia de Funcionários: Receita por Gestor
         Gestor      Funcionario                    Title  Transacoes  NetRevenue  TicketMedio
   (Presidente)    Andrew Fuller    Vice President, Sales         241   166537.75       691.03
  Andrew Fuller Margaret Peacock     Sales Representative         420   232890.85       554.50
  Andrew Fuller  Janet Leverling     Sales Representative         321   202812.84       631.82
  Andrew Fuller    Nancy Davolio     Sales Representative         345   192107.60       556.83
  Andrew Fuller   Laura Callahan Inside Sales Coordinator         260   126862.28       487.93
  Andrew Fuller  Steven Buchanan            Sales Manager         117    68792.28       587.97
Steven Buchanan      Robert King     Sales Representative         176   124568.23       707.77
Steven Buchanan   Anne Dodsworth     Sales Representative         107    77308.07       722.51
Steven Buchanan   Michael Suyama     Sales Representative         168    73913.13       

In [8]:
# ============================================================
# Query 7: Sazonalidade com LAG — variação MoM (Month-over-Month)
# Técnica DuckDB: LAG para comparar mês atual com anterior
# ============================================================
print("Query 7 — Sazonalidade: NetRevenue por Mês com variação MoM (LAG)")
print(conn.execute("""
    SELECT
        d.Year,
        d.Quarter,
        d.Month,
        d.MonthName,
        COUNT(DISTINCT f.OrderID)                               AS Pedidos,
        ROUND(SUM(f.NetRevenue), 2)                             AS NetRevenue,
        ROUND(SUM(SUM(f.NetRevenue)) OVER (
            PARTITION BY d.Year, d.Quarter
            ORDER BY d.Month
        ), 2)                                                   AS AcumuladoTrimestre,
        ROUND(SUM(f.NetRevenue) - LAG(SUM(f.NetRevenue)) OVER (
            ORDER BY d.Year, d.Month
        ), 2)                                                   AS VarMoM
    FROM gold.FactSales f
    JOIN gold.DimDate d ON f.OrderDateKey = d.DateKey
    GROUP BY d.Year, d.Quarter, d.Month, d.MonthName
    ORDER BY d.Year, d.Month
""").fetchdf().to_string(index=False))

Query 7 — Sazonalidade: NetRevenue por Mês com variação MoM (LAG)
 Year  Quarter  Month MonthName  Pedidos  NetRevenue  AcumuladoTrimestre     VarMoM
 1996        3      7      July       22    27861.89            27861.89        NaN
 1996        3      8    August       25    25485.27            53347.17   -2376.62
 1996        3      9 September       23    26381.40            79728.57     896.13
 1996        4     10   October       26    37515.72            37515.72   11134.32
 1996        4     11  November       25    45600.04            83115.77    8084.32
 1996        4     12  December       31    45239.63           128355.40    -360.41
 1997        1      1   January       33    61258.07            61258.07   16018.44
 1997        1      2  February       29    38483.63            99741.70  -22774.43
 1997        1      3     March       30    38547.22           138288.92      63.59
 1997        2      4     April       31    53032.95            53032.95   14485.73
 1997     

In [9]:
# ============================================================
# Query 8: Produtos com reposição necessária (último snapshot)
# ============================================================
print("Query 8 — Produtos que Precisam de Reposição (Periodic Snapshot)")
print(conn.execute("""
    SELECT
        p.ProductName,
        p.CategoryName,
        fs.UnitsInStock,
        fs.ReorderLevel,
        fs.UnitsOnOrder,
        fs.UnitsInStock - fs.ReorderLevel   AS EstoqueAcimaReorder,
        fs.SnapshotDateKey
    FROM gold.FactProductStock fs
    JOIN gold.DimProduct p ON fs.ProductSK = p.ProductSK
    WHERE fs.NeedsReorder = TRUE
      AND fs.SnapshotDateKey = (SELECT MAX(SnapshotDateKey) FROM gold.FactProductStock)
    ORDER BY EstoqueAcimaReorder
""").fetchdf().to_string(index=False))

Query 8 — Produtos que Precisam de Reposição (Periodic Snapshot)


              ProductName   CategoryName  UnitsInStock  ReorderLevel  UnitsOnOrder  EstoqueAcimaReorder  SnapshotDateKey
        Gorgonzola Telino Dairy Products             0            20            70                  -20         20260329
       Mascarpone Fabioli Dairy Products             9            25            40                  -16         20260329
Louisiana Hot Spiced Okra     Condiments             4            20           100                  -16         20260329
            Outback Lager      Beverages            15            30            10                  -15         20260329
               Gravad lax        Seafood            11            25            50                  -14         20260329
            Aniseed Syrup     Condiments            13            25            70                  -12         20260329
              Rogede sild        Seafood             5            15            70                  -10         20260329
                Chocolade    Con

In [10]:
# ============================================================
# Query 9: Desconto médio por categoria + NTILE para segmentação
# Técnica DuckDB: NTILE para classificar categorias por desconto
# ============================================================
print("Query 9 — Impacto do Desconto por Categoria (com NTILE)")
print(conn.execute("""
    SELECT
        p.CategoryName,
        ROUND(AVG(f.Discount) * 100, 2)                         AS DescontoMedioPct,
        ROUND(SUM(f.GrossRevenue), 2)                           AS GrossRevenue,
        ROUND(SUM(f.NetRevenue), 2)                             AS NetRevenue,
        ROUND(SUM(f.GrossRevenue - f.NetRevenue), 2)            AS ReceitaPerdida,
        ROUND(100.0 * SUM(f.GrossRevenue - f.NetRevenue)
              / SUM(f.GrossRevenue), 2)                         AS PctPerdidaPorDesconto,
        -- NTILE: segmenta as 8 categorias em quartis por desconto médio
        NTILE(4) OVER (ORDER BY AVG(f.Discount))                AS QuartilDesconto
    FROM gold.FactSales f
    JOIN gold.DimProduct p ON f.ProductSK = p.ProductSK
    GROUP BY p.CategoryName
    ORDER BY PctPerdidaPorDesconto DESC
""").fetchdf().to_string(index=False))

Query 9 — Impacto do Desconto por Categoria (com NTILE)
  CategoryName  DescontoMedioPct  GrossRevenue  NetRevenue  ReceitaPerdida  PctPerdidaPorDesconto  QuartilDesconto
  Meat/Poultry              6.45     178188.80   163022.36        15166.44                   8.51                4
       Seafood              6.02     141623.09   131261.74        10361.35                   7.32                3
    Condiments              5.26     113694.75   106047.08         7647.67                   6.73                2
Dairy Products              5.34     251330.50   234507.28        16823.22                   6.69                2
     Beverages              6.19     286526.95   267868.18        18658.77                   6.51                4
   Confections              5.69     177099.10   167357.22         9741.88                   5.50                3
       Produce              4.54     105268.60    99984.58         5284.02                   5.02                1
Grains/Cereals          

In [11]:
# ============================================================
# Query 10: Pedidos por país de destino com lead time médio
# Usa FactOrderFulfillment — Accumulating Snapshot
# ============================================================
print("Query 10 — Pedidos por País de Destino (Lead Time + Pontualidade)")
print(conn.execute("""
    SELECT
        f.ShipCountry,
        COUNT(DISTINCT f.OrderID)                                       AS TotalPedidos,
        SUM(CASE WHEN f.IsLate = FALSE THEN 1 ELSE 0 END)               AS Pontuais,
        SUM(CASE WHEN f.IsLate = TRUE  THEN 1 ELSE 0 END)               AS Atrasados,
        ROUND(AVG(f.DaysToShip::DOUBLE), 1)                            AS LeadTimeMedio,
        ROUND(100.0 * SUM(CASE WHEN f.IsLate = FALSE THEN 1 ELSE 0 END)
              / NULLIF(COUNT(CASE WHEN f.ShippedDateKey IS NOT NULL THEN 1 END), 0),
              1)                                                        AS OnTimePct
    FROM gold.FactOrderFulfillment f
    GROUP BY f.ShipCountry
    ORDER BY TotalPedidos DESC
""").fetchdf().to_string(index=False))

Query 10 — Pedidos por País de Destino (Lead Time + Pontualidade)
ShipCountry  TotalPedidos  Pontuais  Atrasados  LeadTimeMedio  OnTimePct
    Germany           122       116          4            8.0       96.7
        USA           122       112          7            9.6       94.1
     Brazil            83        78          3            8.1       96.3
     France            77        72          3            8.4       96.0
         UK            56        52          4            8.2       92.9
  Venezuela            46        41          2            8.5       95.3
    Austria            40        37          2          269.9       94.9
     Sweden            37        34          3           10.2       91.9
     Canada            30        29          0            5.9      100.0
     Mexico            28        27          0            7.8      100.0
      Italy            28        25          2            7.9       92.6
      Spain            23        22          1            

In [12]:
# ============================================================
# BÔNUS: PIVOT nativo do DuckDB
# Receita por categoria e ano em formato pivotado
# ============================================================
print("BÔNUS — PIVOT nativo DuckDB: Receita por Categoria x Ano")
print(conn.execute("""
    PIVOT (
        SELECT
            p.CategoryName,
            d.Year,
            ROUND(SUM(f.NetRevenue), 2) AS NetRevenue
        FROM gold.FactSales f
        JOIN gold.DimProduct p ON f.ProductSK = p.ProductSK
        JOIN gold.DimDate d    ON f.OrderDateKey = d.DateKey
        GROUP BY p.CategoryName, d.Year
    )
    ON Year
    USING SUM(NetRevenue)
    GROUP BY CategoryName
    ORDER BY CategoryName
""").fetchdf().to_string(index=False))

conn.close()

BÔNUS — PIVOT nativo DuckDB: Receita por Categoria x Ano


  CategoryName     1996      1997      1998
     Beverages 47919.00 103924.30 116024.87
    Condiments 17900.38  55368.59  32778.11
   Confections 29685.55  82657.75  55013.92
Dairy Products 40980.45 115387.64  78139.19
Grains/Cereals  9507.92  56871.82  29364.84
  Meat/Poultry 28813.66  80975.11  53233.59
       Produce 13885.78  54940.77  31158.03
       Seafood 19391.22  66959.22  44911.29